In [1]:
# Assign the RR to PWM PM2.5 from RR curves

In [2]:
import os
import xarray as xr
import numpy as np
from utils.utils import get_scenario_config

In [3]:
# === Health variables ===
# COPD, DIABETES, ISCHEMIC_HEART_DISEASE, LOWER_RESPIRATORY_INFECTIONS, LUNG_CANCER, STROKE
# resp_copd, t2_dm, cvd_ihd, lri, neo_lung, cvd_stroke
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER", "STROKE"]

In [4]:
# === Path config ===
RR_DIR = "/glade/work/awells/air_quality/GBD21/RR_curves/"

In [6]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

PM_DIR = f"/glade/work/awells/air_quality/{model}/pm25/exposure/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/RR/"

n_samples = 1000

for health_VAR in health_vars:
    for ens_num in ensemble_members:
        print(f"Processing {scenario} ensemble member {ens_num:02d} for {health_VAR}")

        dates = f"{years.start}-{years.stop}"

        pm25_file = f"Annual_PM25_country_population_weighted_exposure_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        pm25_path = os.path.join(PM_DIR, pm25_file)
        pm25 = xr.open_dataarray(pm25_path)
        pm25 = pm25.drop_vars("region")

        RR_file = f"IHME_GBD_2021_AIR_POLLUTION_1990_2021_PM_RR_{health_VAR}.nc"
        RR_path = os.path.join(RR_DIR, RR_file)
        RR_list = xr.open_dataset(RR_path)

        RR = RR_list.sel(exposure=pm25, method="nearest")
        # Extract components
        RR_mean = RR["mean"]  # mean
        RR_lower = RR["lower"]  # lower bound of 95% CI
        RR_upper = RR["upper"]  # upper bound of 95% CI

        # Estimate standard deviation from the 95% confidence interval
        # z-score for 97.5% in normal dist ≈ 1.96
        RR_std = (RR_upper - RR_lower) / (2 * 1.96)

        # Now draw n_samples from a normal distribution for each [country, year]
        samples = np.random.normal(
            loc=RR_mean.values[..., np.newaxis],       # mean
            scale=RR_std.values[..., np.newaxis],      # std
            size=(len(pm25.country), len(pm25.year), n_samples)
        )

        # Convert to xarray
        RR_samples = xr.DataArray(
            samples,
            dims=("country", "year", "sample"),
            coords={
                "country": RR.country,
                "year": RR.year,
                "sample": np.arange(n_samples)
            }
        )

        description = (f"Country level Relative Risk value for {scenario} "
                       f"ensemble member {ens_num:02d} using GBD21 RR curves "
                       " - scripts by A.F. Wells (2025)")
        RR_samples.attrs["scenario"] = scenario
        RR_samples.attrs["model"] = model
        RR_samples.attrs["ensemble"] = ens_num
        RR_samples.attrs["description"] = description

        out_file = f"RR_{health_VAR}_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)
        RR_samples.to_netcdf(out_path)

print("All processing complete.")

Processing SSP245_G6 ensemble member 01 for COPD
Processing SSP245_G6 ensemble member 02 for COPD
Processing SSP245_G6 ensemble member 03 for COPD
Processing SSP245_G6 ensemble member 01 for DIABETES
Processing SSP245_G6 ensemble member 02 for DIABETES
Processing SSP245_G6 ensemble member 03 for DIABETES
Processing SSP245_G6 ensemble member 01 for ISCHEMIC_HEART_DISEASE
Processing SSP245_G6 ensemble member 02 for ISCHEMIC_HEART_DISEASE
Processing SSP245_G6 ensemble member 03 for ISCHEMIC_HEART_DISEASE
Processing SSP245_G6 ensemble member 01 for LOWER_RESPIRATORY_INFECTIONS
Processing SSP245_G6 ensemble member 02 for LOWER_RESPIRATORY_INFECTIONS
Processing SSP245_G6 ensemble member 03 for LOWER_RESPIRATORY_INFECTIONS
Processing SSP245_G6 ensemble member 01 for LUNG_CANCER
Processing SSP245_G6 ensemble member 02 for LUNG_CANCER
Processing SSP245_G6 ensemble member 03 for LUNG_CANCER
Processing SSP245_G6 ensemble member 01 for STROKE
Processing SSP245_G6 ensemble member 02 for STROKE
Proc